In [0]:
df = spark.read.csv("/Volumes/workspace/default/lab/netflix_titles.csv", header=True, inferSchema=True)

df.show(20)

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [0]:
spark.read.option("header",True) \
    .csv("/Volumes/workspace/default/lab/netflix_titles.csv") \
    .createOrReplaceTempView("netflix")

In [0]:
##1 

df.groupBy("type").count().show()


+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2676|
|        Movie| 6131|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



In [0]:
## 2. Count of null/missing `director` values.
df.filter(df.director.isNull()).count()

df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [0]:

from pyspark.sql.functions import col

df.filter(col("director").isNull()).count()

missing_director = df.filter(col("director").isNull()).count()

print(missing_director)

2636


In [0]:
df.filter(df["director"].isNull()).count()

md = df.filter(df["director"].isNull()).count()

print(md)

2636


In [0]:
##3. Count of titles per `rating`, sorted descending.

df.groupBy("rating") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+------------+-----+
|      rating|count|
+------------+-----+
|       TV-MA| 3195|
|       TV-14| 2158|
|       TV-PG|  862|
|           R|  796|
|       PG-13|  489|
|       TV-Y7|  334|
|        TV-Y|  307|
|          PG|  286|
|        TV-G|  220|
|          NR|   80|
|           G|   41|
|        NULL|    6|
|    TV-Y7-FV|    6|
|          UR|    3|
|       NC-17|    3|
|        2021|    2|
|        2019|    1|
|        2017|    1|
| Jide Kosoko|    1|
|        2006|    1|
+------------+-----+
only showing top 20 rows


In [0]:
### 4 Top 10 `listed_in` genres by title count 

df.groupBy("listed_in") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10).show()

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|Dramas, Internati...|  361|
|       Documentaries|  358|
|     Stand-Up Comedy|  334|
|Comedies, Dramas,...|  273|
|Dramas, Independe...|  252|
|            Kids' TV|  220|
|Children & Family...|  215|
|Children & Family...|  201|
|Documentaries, In...|  186|
|Dramas, Internati...|  180|
+--------------------+-----+



In [0]:
### 5
df.groupBy("listed_in") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(15).show()
    

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|Dramas, Internati...|  361|
|       Documentaries|  358|
|     Stand-Up Comedy|  334|
|Comedies, Dramas,...|  273|
|Dramas, Independe...|  252|
|            Kids' TV|  220|
|Children & Family...|  215|
|Children & Family...|  201|
|Documentaries, In...|  186|
|Dramas, Internati...|  180|
|Comedies, Interna...|  176|
|Comedies, Interna...|  152|
|              Dramas|  138|
|Dramas, Internati...|  134|
|Action & Adventur...|  132|
+--------------------+-----+



In [0]:
##6. Type of available content
df.select("type").distinct().show()

+-------------+
|         type|
+-------------+
|      TV Show|
|        Movie|
|         NULL|
|William Wyler|
+-------------+



In [0]:
## 7 

from pyspark.sql.functions import expr, date_format, col

df.withColumn("date_added", expr("try_to_date(trim(date_added), 'MMMM d, yyyy')")) \
    .filter(col("date_added").isNotNull()) \
    .groupBy(date_format("date_added", "yyyy-MM").alias("date")) \
    .count() \
    .orderBy("date") \
    .show()

+-------+-----+
|   date|count|
+-------+-----+
|2008-01|    1|
|2008-02|    1|
|2009-05|    1|
|2009-11|    1|
|2010-11|    1|
|2011-05|    1|
|2011-09|    1|
|2011-10|   11|
|2012-02|    1|
|2012-11|    1|
|2012-12|    1|
|2013-03|    1|
|2013-08|    1|
|2013-09|    2|
|2013-10|    3|
|2013-11|    2|
|2013-12|    2|
|2014-01|    2|
|2014-02|    2|
|2014-04|    2|
+-------+-----+
only showing top 20 rows


In [0]:
##8

from pyspark.sql.functions import split, avg, col, expr

tv_shows = df.filter(df.type == "TV Show")

tv_shows = tv_shows.withColumn(
    "seasons",
    split(tv_shows.duration, " ")[0]
)

tv_shows.withColumn(
    "seasons",
    expr("try_cast(seasons as int)")
).select(avg("seasons").alias("average_seasons")
).show()

+------------------+
|   average_seasons|
+------------------+
|1.7654320987654322|
+------------------+



In [0]:
###9 

df.filter(df["title"].contains("Christmas")).show()



+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|       date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+
|  s1517|TV Show|  Home for Christmas|                NULL|Ida Elise Broch, ...|              Norway|December 18, 2020|        2020| TV-MA|2 Seasons|International TV ...|Tired of the cons...|
|  s1524|  Movie|An Unremarkable C...|  Juan Camilo Pinzon|Antonio Sanint, L...|            Colombia|December 17, 2020|        2020| TV-14|   83 min|Comedies, Dramas,...|An accountant and...|
|  s1536|TV Show|How To Ruin Chris...|  

In [0]:
##10

df.filter(df["cast"].contains("Ryan Reynolds")).show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|    s47|Movie|          Safe House|     Daniel Espinosa|Denzel Washington...|South Africa, Uni...|September 16, 2021|        2012|     R| 115 min|  Action & Adventure|Young CIA operati...|
|   s144|Movie|       Green Lantern|     Martin Campbell|Ryan Reynolds, Bl...|       United States| September 1, 2021|        2011| PG-13| 114 min|Action & Adventur...|Test pilot Hal Jo...|
|  s3032|Movie|Betty White: Firs...|     Steve Boe

In [0]:
##11 

covid_time = df.filter((df["release_year"] == "2020") | (df["release_year"] == "2021"))

covid_time.groupBy("type").count().show()

+-------+-----+
|   type|count|
+-------+-----+
|TV Show|  750|
|  Movie|  791|
+-------+-----+



In [0]:
##12 

from pyspark.sql.functions import split, expr

movies =  df.filter(df["type"] == "Movie")

movies = movies.withColumn("minutes", expr("try_cast(split(duration, ' ')[0] as int)"))
movies.filter(
    (movies["minutes"] > 89) &
    (movies["minutes"] < 121)
).show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+-------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|minutes|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+-------+
|     s1|Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|  90 min|       Documentaries|As her father nea...|     90|
|     s7|Movie|My Little Pony: A...|Robert Cullen, Jo...|Vanessa Hudgens, ...|                NULL|September 24, 2021|        2021|    PG|  91 min|Children & Family...|Equestria's divid...|     91|
|    s10|M